# 01. 링크·지연 기초 실습

목표: dB, 자유공간 경로 손실(FSPL), 링크 마진, 왕복 시간(RTT), 대역폭-지연 곱(BDP)을 계산한다. 모든 모델은 교육용이며 실제 링크 검증을 대체하지 않는다.

In [1]:
import math

C_M_S = 299_792_458.0  # 진공에서의 빛의 속도[m/s]
BOLTZMANN_DBW_K_HZ = -228.6

def db_to_ratio(db):
    return 10 ** (db / 10)

def ratio_to_db(ratio):
    if ratio <= 0:
        raise ValueError('비율은 양수여야 합니다.')
    return 10 * math.log10(ratio)

def dbm_to_watt(dbm):
    return 10 ** ((dbm - 30) / 10)

for value in (0, 10, 20, 30):
    print(f'{value:2d} dBm = {dbm_to_watt(value):.6f} W')

 0 dBm = 0.001000 W
10 dBm = 0.010000 W
20 dBm = 0.100000 W
30 dBm = 1.000000 W


In [2]:
def propagation_delay(distance_km):
    return distance_km * 1000 / C_M_S

scenarios = {
    'LEO 직하점 600 km 한 구간': 600,
    'GEO 한 구간': 35_786,
    '평균 지구-달': 384_400,
}
for name, distance in scenarios.items():
    one_way = propagation_delay(distance)
    print(f'{name}: 편도 {one_way:.4f} s, 같은 경로 왕복 {2*one_way:.4f} s')

# 주의: 실제 지상 단말 간 위성 경로는 uplink와 downlink를 모두 포함한다.

LEO 직하점 600 km 한 구간: 편도 0.0020 s, 같은 경로 왕복 0.0040 s
GEO 한 구간: 편도 0.1194 s, 같은 경로 왕복 0.2387 s
평균 지구-달: 편도 1.2822 s, 같은 경로 왕복 2.5644 s


In [3]:
def fspl_db(distance_km, frequency_ghz):
    if distance_km <= 0 or frequency_ghz <= 0:
        raise ValueError('거리와 주파수는 양수여야 합니다.')
    return 92.45 + 20 * math.log10(distance_km) + 20 * math.log10(frequency_ghz)

def rf_link_budget(eirp_dbw, distance_km, frequency_ghz, g_over_t_db_k,
                   bitrate_bps, required_ebn0_db, extra_losses_db=0, implementation_db=0):
    path_loss = fspl_db(distance_km, frequency_ghz)
    # C/N0 = EIRP - loss + G/T - k. k가 -228.6이므로 -k는 +228.6이다.
    cn0 = eirp_dbw - path_loss - extra_losses_db + g_over_t_db_k - BOLTZMANN_DBW_K_HZ
    ebn0 = cn0 - 10 * math.log10(bitrate_bps)
    margin = ebn0 - required_ebn0_db - implementation_db
    return {'FSPL_dB': path_loss, 'C/N0_dBHz': cn0, 'Eb/N0_dB': ebn0, 'margin_dB': margin}

case = rf_link_budget(20, 1500, 8.2, 3, 2_000_000, 3.0, extra_losses_db=4, implementation_db=2)
for key, value in case.items():
    print(f'{key:12s}: {value:7.2f}')
assert abs(fspl_db(1, 1) - 92.45) < 1e-9

FSPL_dB     :  174.25
C/N0_dBHz   :   73.35
Eb/N0_dB    :   10.34
margin_dB   :    5.34


In [4]:
def shannon_capacity_bps(bandwidth_hz, snr_db):
    return bandwidth_hz * math.log2(1 + db_to_ratio(snr_db))

def bandwidth_delay_product_bytes(rate_bps, rtt_s):
    return rate_bps * rtt_s / 8

for snr in (-5, 0, 5, 10):
    capacity = shannon_capacity_bps(1_000_000, snr)
    print(f'B=1 MHz, SNR={snr:>3} dB -> Shannon 상한 {capacity/1e6:.3f} Mbit/s')

rate, rtt = 100_000_000, 0.080
bdp = bandwidth_delay_product_bytes(rate, rtt)
print(f'100 Mbit/s × 80 ms BDP = {bdp:,.0f} bytes')
assert bdp == 1_000_000

B=1 MHz, SNR= -5 dB -> Shannon 상한 0.396 Mbit/s
B=1 MHz, SNR=  0 dB -> Shannon 상한 1.000 Mbit/s
B=1 MHz, SNR=  5 dB -> Shannon 상한 2.057 Mbit/s
B=1 MHz, SNR= 10 dB -> Shannon 상한 3.459 Mbit/s
100 Mbit/s × 80 ms BDP = 1,000,000 bytes


## 연습

1. `bitrate_bps`를 두 배로 바꾸고 Eb/N0와 margin 변화를 설명한다.
2. 거리와 주파수를 각각 두 배로 바꾸어 FSPL이 약 6.02 dB 증가하는지 확인한다.
3. link margin이 0 dB보다 크다는 사실만으로 availability가 보장되지 않는 이유를 다섯 가지 적는다.